# GenEval-Only Notebook

This notebook contains only the GenEval pipeline in 3 cells:
1. Setup and model/critic loading
2. GenEval helper functions
3. GenEval execution and comparison plotting

In [ ]:
import os
import json
import math
import glob
import re
import subprocess
import sys
from collections import defaultdict

import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import open_clip

from modeling.tatitok import TATiTok
from modeling.maskgen import MaskGen_VQ
from token_regrate.train_token_regret_ddp import (
    set_global_seed,
    load_trained_critic,
    generate_image_vq_batch,
)

HF_CACHE_DIR = os.path.abspath(os.environ.get('HF_CACHE_DIR', 'hf_cache'))
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR
os.environ['OPENCLIP_CACHE_DIR'] = HF_CACHE_DIR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('HF cache:', HF_CACHE_DIR)

def load_pretrained_stack(root_dir=HF_CACHE_DIR):
    tok_dir = os.path.join(root_dir, 'turkeyju/tokenizer_tatitok_bl128_vq')
    gen_dir = os.path.join(root_dir, 'turkeyju/generator_maskgen_vq_xl')

    tatitok_vq_tokenizer = TATiTok.from_pretrained(pretrained_model_name_or_path=tok_dir, cache_dir=tok_dir)
    tatitok_vq_tokenizer.eval()
    tatitok_vq_tokenizer.requires_grad_(False)

    maskgen_vq_generator = MaskGen_VQ.from_pretrained(pretrained_model_name_or_path=gen_dir, cache_dir=gen_dir)
    maskgen_vq_generator.eval()
    maskgen_vq_generator.requires_grad_(False)

    clip_encoder, _, _ = open_clip.create_model_and_transforms(
        'ViT-L-14-336', pretrained='openai', force_quick_gelu=True
    )
    del clip_encoder.visual
    clip_tokenizer = open_clip.get_tokenizer('ViT-L-14-336')
    clip_encoder.transformer.batch_first = False
    clip_encoder.eval()
    clip_encoder.requires_grad_(False)

    return tatitok_vq_tokenizer, maskgen_vq_generator, clip_tokenizer, clip_encoder

tatitok_vq_tokenizer, maskgen_vq_generator, clip_tokenizer, clip_encoder = load_pretrained_stack()
maskgen_vq_generator = maskgen_vq_generator.to(device)
tatitok_vq_tokenizer = tatitok_vq_tokenizer.to(device)
clip_encoder = clip_encoder.to(device)


attention mode is flash
Using device: cuda
HF cache: /home/behzad/MaskGen/hf_cache
Loading weights from local directory
Loading weights from local directory


In [2]:

CRITIC_CKPT_PATH = 'outputs/token_regret_critic/critic_step_190.pt'

critic_head = load_trained_critic(CRITIC_CKPT_PATH, maskgen_vq_generator, use_hidden=True)
critic_head = critic_head.to(next(maskgen_vq_generator.parameters()).device)

print('Loaded critic head from:', CRITIC_CKPT_PATH)

GENEVAL = {
    'root': 'geneval',
    'output_root': 'outputs/geneval_eval',
    'max_prompts': None,
    'batch_size': 4,
    'seed': 42,
    'seed_policy': 'per_prompt_sample',
    'guidance_scale': 12.0,
    'randomize_temperature': 2.0,
    'aesthetic_score': 6.5,
    'num_sample_steps': 16,
    'remask_ratio': 0.5,
    'refine_loops': 20,
    'refine_start_step': 8,
    'run_generation': True,
}


Loaded critic head from: outputs/token_regret_critic/critic_step_190.pt


In [3]:
def _resolve_geneval_paths(geneval_root='geneval'):
    root = os.path.abspath(geneval_root)
    eval_script = os.path.join(root, 'evaluation', 'evaluate_images.py')
    summary_script = os.path.join(root, 'evaluation', 'summary_scores.py')
    metadata_jsonl = os.path.join(root, 'prompts', 'evaluation_metadata.jsonl')
    model_path = os.path.join(root, 'saved_models')
    model_config = os.path.join(
        root,
        'mmdetection',
        'configs',
        'mask2former',
        'mask2former_swin-s-p4-w7-224_lsj_8x2_50e_coco.py',
    )

    required = {
        'geneval_root': root,
        'eval_script': eval_script,
        'summary_script': summary_script,
        'metadata_jsonl': metadata_jsonl,
        'model_path': model_path,
        'model_config': model_config,
    }
    missing = [k for k, v in required.items() if not os.path.exists(v)]
    if missing:
        pretty = '\n'.join(f'- {k}: {required[k]}' for k in missing)
        raise FileNotFoundError('Missing GenEval assets:\n' + pretty)
    return required


def _load_geneval_metadata(metadata_jsonl_path):
    rows = []
    with open(metadata_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    if len(rows) == 0:
        raise RuntimeError('No prompts found in: ' + str(metadata_jsonl_path))
    return rows


def _save_geneval_prompt_samples(prompt_idx, metadata, images, out_root, save_grid=True):
    prompt_dir = os.path.join(out_root, f'{int(prompt_idx):05d}')
    samples_dir = os.path.join(prompt_dir, 'samples')
    os.makedirs(samples_dir, exist_ok=True)

    metadata_path = os.path.join(prompt_dir, 'metadata.jsonl')
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=True)

    for i, img in enumerate(images):
        img.save(os.path.join(samples_dir, f'{int(i):04d}.png'))

    if save_grid and len(images) > 0:
        n = len(images)
        cols = min(4, n)
        rows = math.ceil(n / cols)
        w, h = images[0].size
        grid = Image.new('RGB', (cols * w, rows * h), color=(255, 255, 255))
        for idx, img in enumerate(images):
            r = idx // cols
            c = idx % cols
            grid.paste(img, (c * w, r * h))
        grid.save(os.path.join(prompt_dir, 'grid.png'))


def _prompt_level_summary_from_results(results_jsonl):
    rows = []
    with open(results_jsonl, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    if len(rows) == 0:
        raise RuntimeError('No rows found in result file: ' + str(results_jsonl))

    tag_prompt_any = defaultdict(dict)
    prompt_sample_count = defaultdict(int)
    for r in rows:
        tag = r['tag']
        meta = r['metadata']
        corr = bool(r['correct'])
        if meta not in tag_prompt_any[tag]:
            tag_prompt_any[tag][meta] = False
        tag_prompt_any[tag][meta] = tag_prompt_any[tag][meta] or corr
        prompt_sample_count[meta] += 1

    tag_scores = {}
    for tag, prompt_map in tag_prompt_any.items():
        vals = list(prompt_map.values())
        tag_scores[tag] = float(sum(vals) / max(len(vals), 1))

    overall_prompt_level = float(sum(tag_scores.values()) / max(len(tag_scores), 1))
    return {
        'overall_prompt_level': overall_prompt_level,
        'tag_scores': tag_scores,
        'num_rows': int(len(rows)),
        'num_prompts': int(len(prompt_sample_count)),
    }


def _resolve_mmdet_root(paths):
    preferred = os.path.abspath(os.path.join('geneval', 'mmdetection'))
    fallback = os.path.join(paths['geneval_root'], 'mmdetection')
    for p in [preferred, fallback]:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError('MMDetection root not found.')


def _run_subprocess_stream(cmd, env, log_file):
    os.makedirs(os.path.dirname(log_file), exist_ok=True)
    tail_lines = []
    with open(log_file, 'w', encoding='utf-8') as f:
        proc = subprocess.Popen(
            cmd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if proc.stdout is not None:
            for line in proc.stdout:
                print(line, end='')
                f.write(line)
                tail_lines.append(line)
                if len(tail_lines) > 120:
                    tail_lines = tail_lines[-120:]
            proc.stdout.close()
        return_code = proc.wait()
    return return_code, ''.join(tail_lines)


def run_official_geneval_eval(
    eval_dir,
    out_file,
    geneval_root='geneval',
    eval_python=sys.executable,
    cuda_visible_devices=None,
):
    paths = _resolve_geneval_paths(geneval_root=geneval_root)
    cmd = [
        eval_python,
        paths['eval_script'],
        eval_dir,
        '--outfile',
        out_file,
        '--model-path',
        paths['model_path'],
        '--model-config',
        paths['model_config'],
    ]
    os.makedirs(os.path.dirname(out_file), exist_ok=True)

    env = os.environ.copy()
    mmdet_root = _resolve_mmdet_root(paths)
    base_py = env.get('PYTHONPATH', '')
    env['PYTHONPATH'] = mmdet_root if not base_py else mmdet_root + ':' + base_py
    if cuda_visible_devices is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(cuda_visible_devices)

    stderr_file = out_file.replace('.jsonl', '_eval_stderr.log')
    rc, tail_text = _run_subprocess_stream(cmd, env, stderr_file)
    if rc != 0:
        print(tail_text[-2000:])
        raise RuntimeError(f'Official GenEval evaluation failed. Check: {stderr_file}')

    summary_cmd = [eval_python, paths['summary_script'], out_file]
    summary_run = subprocess.run(summary_cmd, env=env, capture_output=True, text=True)
    paper_protocol = _prompt_level_summary_from_results(out_file)
    paper_protocol['outfile'] = out_file

    summary_json = os.path.splitext(out_file)[0] + '_paper_protocol_summary.json'
    with open(summary_json, 'w', encoding='utf-8') as f:
        json.dump(paper_protocol, f, indent=2)

    if summary_run.returncode == 0:
        print(summary_run.stdout.strip())

    return {
        'results_jsonl': out_file,
        'paper_protocol_summary_json': summary_json,
        'paper_protocol': paper_protocol,
        'summary_scores_stdout': summary_run.stdout,
    }


def generate_geneval_images_maskgen(
    model,
    tokenizer,
    clip_tokenizer,
    clip_encoder,
    metadata_jsonl,
    outdir,
    batch_size=4,
    max_prompts=None,
    seed=42,
    guidance_scale=12.0,
    randomize_temperature=1.5,
    aesthetic_score=6.5,
    num_sample_steps=16,
    use_regret_remask=False,
    critic=None,
    remask_ratio=0.15,
    refine_loops=1,
    refine_start_step=8,
    critic_use_hidden=True,
    seed_policy='per_prompt_sample',
):
    rows = _load_geneval_metadata(metadata_jsonl)
    if max_prompts is not None:
        rows = rows[:max(1, int(max_prompts))]

    os.makedirs(outdir, exist_ok=True)
    set_global_seed(seed)
    seed_policy = str(seed_policy).strip().lower()

    model_device = next(model.parameters()).device
    if next(clip_encoder.parameters()).device != model_device:
        clip_encoder = clip_encoder.to(model_device)
    if next(tokenizer.parameters()).device != model_device:
        tokenizer = tokenizer.to(model_device)

    if use_regret_remask:
        if critic is None:
            raise ValueError('use_regret_remask=True requires critic')
        if next(critic.parameters()).device != model_device:
            critic = critic.to(model_device)
        critic.eval()

    for idx, meta in enumerate(tqdm(rows, desc='GenEval generation')):
        prompt = str(meta.get('prompt', '')).strip()
        if not prompt:
            raise RuntimeError(f'Prompt is empty for metadata row {idx}')

        if seed_policy == 'per_prompt_sample':
            images = []
            for sample_idx in range(int(batch_size)):
                sample_seed = int(seed) + idx * int(batch_size) + sample_idx
                set_global_seed(sample_seed)
                sample_images = generate_image_vq_batch(
                    prompts=[prompt],
                    model=model,
                    tokenizer=tokenizer,
                    clip_tokenizer=clip_tokenizer,
                    clip_encoder=clip_encoder,
                    guidance_scale=float(guidance_scale),
                    randomize_temperature=float(randomize_temperature),
                    aesthetic_score=float(aesthetic_score),
                    num_sample_steps=int(num_sample_steps),
                    use_regret_remask=bool(use_regret_remask),
                    critic=critic,
                    remask_ratio=float(remask_ratio),
                    refine_loops=int(refine_loops),
                    refine_start_step=int(refine_start_step),
                    critic_use_hidden=bool(critic_use_hidden),
                    device=model_device,
                )
                if len(sample_images) != 1:
                    raise RuntimeError('Expected one generated image per sample when seed_policy=per_prompt_sample.')
                images.append(sample_images[0])
        else:
            prompts = [prompt] * int(batch_size)
            images = generate_image_vq_batch(
                prompts=prompts,
                model=model,
                tokenizer=tokenizer,
                clip_tokenizer=clip_tokenizer,
                clip_encoder=clip_encoder,
                guidance_scale=float(guidance_scale),
                randomize_temperature=float(randomize_temperature),
                aesthetic_score=float(aesthetic_score),
                num_sample_steps=int(num_sample_steps),
                use_regret_remask=bool(use_regret_remask),
                critic=critic,
                remask_ratio=float(remask_ratio),
                refine_loops=int(refine_loops),
                refine_start_step=int(refine_start_step),
                critic_use_hidden=bool(critic_use_hidden),
                device=model_device,
            )
        _save_geneval_prompt_samples(idx, meta, images, outdir)

    return outdir


def _has_reusable_geneval_images(image_dir, min_prompts=1, min_samples_per_prompt=1):
    if not os.path.isdir(image_dir):
        return False
    prompt_dirs = []
    for name in os.listdir(image_dir):
        p = os.path.join(image_dir, name)
        if os.path.isdir(p) and name.isdigit():
            prompt_dirs.append(p)
    if len(prompt_dirs) < int(min_prompts):
        return False

    valid_prompt_count = 0
    for pdir in prompt_dirs:
        meta_path = os.path.join(pdir, 'metadata.jsonl')
        samples_dir = os.path.join(pdir, 'samples')
        if not os.path.isfile(meta_path) or not os.path.isdir(samples_dir):
            continue
        sample_files = [
            x for x in os.listdir(samples_dir)
            if x.lower().endswith('.png') and x[:-4].isdigit()
        ]
        if len(sample_files) >= int(min_samples_per_prompt):
            valid_prompt_count += 1

    return valid_prompt_count >= int(min_prompts)


def _load_geneval_paper_protocol_summary(out_file):
    summary_json = os.path.splitext(out_file)[0] + '_paper_protocol_summary.json'
    if not os.path.isfile(summary_json):
        return None
    print(f'Found existing GenEval summary for {out_file}: {summary_json}')
    with open(summary_json, 'r', encoding='utf-8') as f:
        paper_protocol = json.load(f)
    return {
        'results_jsonl': out_file,
        'paper_protocol_summary_json': summary_json,
        'paper_protocol': paper_protocol,
        'summary_scores_stdout': '',
    }


def run_geneval_baseline_vs_head(
    model,
    tokenizer,
    clip_tokenizer,
    clip_encoder,
    critic,
    geneval_root='geneval',
    output_root='outputs/geneval_eval',
    eval_python=sys.executable,
    max_prompts=None,
    batch_size=4,
    seed=42,
    seed_policy='per_prompt_sample',
    guidance_scale=12.0,
    randomize_temperature=1.5,
    aesthetic_score=6.5,
    num_sample_steps=16,
    remask_ratio=0.15,
    refine_loops=1,
    refine_start_step=8,
    critic_use_hidden=True,
    run_generation=True,
):
    paths = _resolve_geneval_paths(geneval_root=geneval_root)
    os.makedirs(output_root, exist_ok=True)

    baseline_image_dir = os.path.join(output_root, 'baseline_images')
    head_image_dir = os.path.join(output_root, 'head_images')

    min_prompts = max(1, int(max_prompts)) if max_prompts is not None else 1
    min_samples = max(1, int(batch_size))
    baseline_ready = _has_reusable_geneval_images(
        baseline_image_dir,
        min_prompts=min_prompts,
        min_samples_per_prompt=min_samples,
    )
    head_ready = _has_reusable_geneval_images(
        head_image_dir,
        min_prompts=min_prompts,
        min_samples_per_prompt=min_samples,
    )

    if not baseline_ready:
        if not bool(run_generation):
            raise RuntimeError('Baseline images missing and run_generation=False.')
        generate_geneval_images_maskgen(
            model=model,
            tokenizer=tokenizer,
            clip_tokenizer=clip_tokenizer,
            clip_encoder=clip_encoder,
            metadata_jsonl=paths['metadata_jsonl'],
            outdir=baseline_image_dir,
            batch_size=int(batch_size),
            max_prompts=max_prompts,
            seed=int(seed),
            seed_policy=seed_policy,
            guidance_scale=float(guidance_scale),
            randomize_temperature=float(randomize_temperature),
            aesthetic_score=float(aesthetic_score),
            num_sample_steps=int(num_sample_steps),
            use_regret_remask=False,
            critic=None,
        )

    if not head_ready:
        if not bool(run_generation):
            raise RuntimeError('Head images missing and run_generation=False.')
        generate_geneval_images_maskgen(
            model=model,
            tokenizer=tokenizer,
            clip_tokenizer=clip_tokenizer,
            clip_encoder=clip_encoder,
            metadata_jsonl=paths['metadata_jsonl'],
            outdir=head_image_dir,
            batch_size=int(batch_size),
            max_prompts=max_prompts,
            seed=int(seed),
            seed_policy=seed_policy,
            guidance_scale=float(guidance_scale),
            randomize_temperature=float(randomize_temperature),
            aesthetic_score=float(aesthetic_score),
            num_sample_steps=int(num_sample_steps),
            use_regret_remask=True,
            critic=critic,
            remask_ratio=float(remask_ratio),
            refine_loops=int(refine_loops),
            refine_start_step=int(refine_start_step),
            critic_use_hidden=bool(critic_use_hidden),
        )

    baseline_results = os.path.join(output_root, 'baseline_results.jsonl')
    head_results = os.path.join(output_root, 'head_results.jsonl')

    baseline_eval = _load_geneval_paper_protocol_summary(baseline_results)
    if baseline_eval is None:
        baseline_eval = run_official_geneval_eval(
            eval_dir=baseline_image_dir,
            out_file=baseline_results,
            geneval_root=geneval_root,
            eval_python=eval_python,
        )

    head_eval = _load_geneval_paper_protocol_summary(head_results)
    if head_eval is None:
        head_eval = run_official_geneval_eval(
            eval_dir=head_image_dir,
            out_file=head_results,
            geneval_root=geneval_root,
            eval_python=eval_python,
        )

    baseline_overall = baseline_eval['paper_protocol']['overall_prompt_level']
    head_overall = head_eval['paper_protocol']['overall_prompt_level']

    comparison = {
        'baseline_overall_prompt_level': baseline_overall,
        'head_overall_prompt_level': head_overall,
        'delta_head_minus_baseline': float(head_overall - baseline_overall),
        'baseline_tag_scores': baseline_eval['paper_protocol']['tag_scores'],
        'head_tag_scores': head_eval['paper_protocol']['tag_scores'],
    }

    comparison_path = os.path.join(output_root, 'baseline_vs_head_comparison.json')
    with open(comparison_path, 'w', encoding='utf-8') as f:
        json.dump(comparison, f, indent=2)

    return {
        'baseline': baseline_eval,
        'head': head_eval,
        'comparison': comparison,
        'comparison_path': comparison_path,
        'baseline_image_dir': baseline_image_dir,
        'head_image_dir': head_image_dir,
    }


def _resolve_geneval_eval_python():
    env_override = str(os.environ.get('GENEVAL_EVAL_PYTHON', '')).strip()
    candidates = [
        env_override,
        '/home/behzad/anaconda3/envs/geneval/bin/python',
        os.path.expanduser('~/anaconda3/envs/geneval/bin/python'),
        os.path.expanduser('~/miniconda3/envs/geneval/bin/python'),
    ]

    seen = set()
    for cand in candidates:
        cand = str(cand).strip()
        if not cand:
            continue

        py = os.path.abspath(os.path.expanduser(cand))
        if py in seen:
            continue
        seen.add(py)

        if not os.path.isfile(py) or not os.access(py, os.X_OK):
            continue

        probe = subprocess.run(
            [py, '-c', 'import mmcv, mmdet; print(mmcv.__version__); print(mmdet.__version__)'],
            capture_output=True,
            text=True,
        )
        if probe.returncode == 0:
            return py

    raise RuntimeError('Could not resolve a valid geneval evaluator python.')


GENEVAL_EVAL_PYTHON = _resolve_geneval_eval_python()

baseline_head_geneval_report = run_geneval_baseline_vs_head(
    model=maskgen_vq_generator,
    tokenizer=tatitok_vq_tokenizer,
    clip_tokenizer=clip_tokenizer,
    clip_encoder=clip_encoder,
    critic=critic_head,
    geneval_root=GENEVAL['root'],
    output_root=GENEVAL['output_root'],
    eval_python=GENEVAL_EVAL_PYTHON,
    max_prompts=GENEVAL['max_prompts'],
    batch_size=GENEVAL['batch_size'],
    seed=GENEVAL['seed'],
    seed_policy=GENEVAL['seed_policy'],
    guidance_scale=GENEVAL['guidance_scale'],
    randomize_temperature=GENEVAL['randomize_temperature'],
    aesthetic_score=GENEVAL['aesthetic_score'],
    num_sample_steps=GENEVAL['num_sample_steps'],
    remask_ratio=GENEVAL['remask_ratio'],
    refine_loops=GENEVAL['refine_loops'],
    refine_start_step=GENEVAL['refine_start_step'],
    critic_use_hidden=True,
    run_generation=GENEVAL['run_generation'],
)


comparison = baseline_head_geneval_report.get('comparison', {})
baseline_tag_scores = comparison.get('baseline_tag_scores', {})
head_tag_scores = comparison.get('head_tag_scores', {})

baseline_overall = comparison.get('baseline_overall_prompt_level')
head_overall = comparison.get('head_overall_prompt_level')
if baseline_overall is None or head_overall is None:
    raise RuntimeError('baseline_head_geneval_report does not contain overall comparison scores.')


GenEval generation:  24%|██▍       | 133/553 [19:52<1:02:46,  8.97s/it]


KeyboardInterrupt: 

## Plot comparison

In [ ]:

tag_order = ['single_object', 'two_object', 'counting', 'colors', 'position', 'color_attr']
label_map = {
    'single_object': 'S. Obj.',
    'two_object': 'T. Obj.',
    'counting': 'Count.',
    'colors': 'Colors',
    'position': 'Position',
    'color_attr': 'C. Attri.',
}
tag_names = [tag for tag in tag_order if tag in baseline_tag_scores or tag in head_tag_scores]
labels = [label_map[tag] for tag in tag_names] + ['Overall']
baseline_values = [float(baseline_tag_scores.get(tag, np.nan)) for tag in tag_names] + [float(baseline_overall)]
head_values = [float(head_tag_scores.get(tag, np.nan)) for tag in tag_names] + [float(head_overall)]

x = np.arange(len(labels))
width = 0.38
fig, ax = plt.subplots(figsize=(max(10, 1.1 * len(labels)), 5.5))
ax.bar(x - width / 2, baseline_values, width, label='Baseline', color='#4C78A8')
ax.bar(x + width / 2, head_values, width, label='With head', color='#F58518')
ax.set_title('GenEval comparison: overall and per category')
ax.set_ylabel('Score')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylim(0, max(1.0, np.nanmax([np.nanmax(baseline_values), np.nanmax(head_values)]) + 0.08))
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False)

for bars in ax.containers:
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)

comparison_path = baseline_head_geneval_report.get('comparison_path')
if comparison_path:
    plot_path = os.path.join(os.path.dirname(comparison_path), 'baseline_vs_head_comparison.png')
    fig.savefig(plot_path, dpi=200, bbox_inches='tight')
    print('Saved comparison plot:', plot_path)

plt.tight_layout()
plt.show()